# Phase 4: Uplift Model

In this notebook we build an uplift model. It takes a customer's features as input, together with the treatment group, and outputs the expected outcome for that customer in that treatment.

### In formula
For a customer with features $x$ (recency, past spend, ...) there are two *potential outcomes*: $Y(1)$, whether they visit if emailed, and $Y(0)$, whether they visit if not emailed. Only one of the two is ever observed, so the effect on a single customer cannot be measured. What we can estimate is the average effect among customers with the same features, the **uplift** (also called CATE, *conditional average treatment effect*):

$$\tau(x) = E\left[\,Y(1) - Y(0) \mid X = x\,\right]$$

- $\tau(x)$: the uplift for customers with features $x$, here the extra probability of visiting caused by the email.
- $E[\,\cdot \mid X = x\,]$: the average over all customers whose features equal $x$.

Because the treatments were assigned at random, emailed and non-emailed customers with the same features are comparable, so the uplift is the difference between two visit probabilities:

$$\tau(x) = \mu_1(x) - \mu_0(x), \qquad \mu_t(x) = P(\text{visit} = 1 \mid T = t,\ X = x)$$

- $T$: the treatment, $T = 1$ for the men's email and $T = 0$ for no email.
- $\mu_t(x)$: the visit probability of a customer with features $x$ in treatment group $t$. This is the model output described above.

A **T-learner** ("T" for *two*) estimates $\mu_1$ and $\mu_0$ with two separate models, one per treatment group, and subtracts their predictions:

$$\hat\tau(x) = \hat\mu_1(x) - \hat\mu_0(x)$$

The hat ( $\hat{}$ ) marks a quantity estimated from the data.

## The targeting rule
Phase 3 found one segment effect (buyer type). Here we ask whether a model can rank *individual* customers by how much the email changes their behaviour, and turn that ranking into a rule: "email the top X%".

- **Comparison:** men's email vs no email (≈ 42,600 customers, half emailed).
- **Model:** a T-learner made of two logistic regressions (a GLM, *generalised linear model*), trained on `visit`.
- **Evaluation:** only on customers each model never saw (cross-fitting), with Qini curves against three baselines: random targeting, a response model, and the phase 3 rule ("Both" buyers first).
- **Economics:** profit for a range of email costs, and where emailing stops paying (break-even).

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.formula.api as smf
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import StratifiedKFold

from digital_sales.experiment import load_analysis_base, proportion_test, values
from digital_sales.segments import load_segments

SEED = 2008
GREY = "#9b9a95"

In [2]:
df = load_analysis_base()
df = df[df["treatment"].isin(["mens_email", "no_email"])].reset_index(drop=True)
# we add buyer type for the baseline rule 'send first to Both buyer type'
df = df.merge(load_segments()[["customer_id", "buyer_type"]], on="customer_id", how="left")
df["t"] = (df["treatment"] == "mens_email").astype(int)        # 1 = got the men's email
df["both"] = (df["buyer_type"] == "Both").astype(int)          # 1 = bought men's AND women's merchandise last year
df = df.astype({"history": float, "spend": float})
print(f"{len(df):,} customers, {df['t'].mean():.0%} emailed, {df['both'].mean():.0%} 'Both' buyers")

42,613 customers, 50% emailed, 10% 'Both' buyers


## T-learner with two logistic regressions
Each of the two models is a logistic regression:

$$\mu_t(x) = \sigma\left(\beta_{t,0} + \beta_{t,1}\,\text{recency} + \beta_{t,2}\,\log(\text{history}) + \beta_{t,3}\,\text{both}\right), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

- $\beta_{t,0}, \dots, \beta_{t,3}$ (beta): the coefficients of the model for treatment group $t$, on the log-odds scale. Each group gets its own set: that is what makes this a T-learner.
- $\sigma$ (sigma): the logistic function. It turns any number $z$ into a probability between 0 and 1.
- `recency`: months since the last purchase (1-12). `history`: last year's spend in \$, logged because a few big spenders form a long right tail. `both`: 1 for customers who bought men's *and* women's merchandise.

**Why so few features?** A T-learner estimates every coefficient twice, and the uplift is the difference of two noisy models, so every extra feature adds noise to the uplift. We keep three: recency and spend (the R and M of RFM, the classic direct-marketing variables *recency, frequency, monetary value*), plus the "Both" flag, the one segment where phase 3 found a different email effect. This set was picked after a quick check of a few alternatives (more features, and a LightGBM version with all seven features, did worse), so the presented model was already chosen to be more optimal than some other alternatives.  

**Cross-fitting:** the data is split into 5 folds; each fold is scored by models trained on the other 4. Every customer gets a score from models that never saw them, so all the evaluation below is out-of-sample. Because the score uses only pre-campaign features, emailed and non-emailed customers stay comparable within any score band (randomisation), which is what makes the evaluation valid.

**Ranking within folds:** each fold is scored by its own pair of models, so scores from different folds sit on slightly different scales. A fold whose held-out customers happened to respond well leaves weaker training data behind, and therefore gets lower scores; pooling the raw scores would push those good responders down the ranking. So we rank each customer by their percentile *within* their fold (0 = lowest score in the fold, 1 = highest). The buyer-type rule is not fitted, so it needs no such step.

In [3]:
# --- Model -------------------------------------------------------------------
# FORMULA: the logistic regression fitted to each treatment group (statsmodels formula syntax, outcome ~ features)
#   visit            outcome: 1 = visited the website in the two weeks after the campaign, 0 = not
#   recency          months since the last purchase (1-12)
#   np.log(history)  log of last year's spend
#   both             1 = "Both" buyer, 0 = men's only or women's only
FORMULA = "visit ~ recency + np.log(history) + both"

# --- Cross-fitting setup ---------------------------------------------------
# folds: splits the customers into 5 groups ("folds"). Each round trains on 4 folds and scores the 5th.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# strata: one label per customer combining treatment and outcome ("1_1", "1_0", "0_1", "0_0").
#         StratifiedKFold balances this label, so every fold has the same emailed/visited mix.
strata = df["t"].astype(str) + "_" + df["visit"].astype(str)

# is_treated: True/False array, True for customers who got the men's email.
is_treated = df["t"].to_numpy() == 1

# mu1:  predicted visit probability IF emailed, one value per customer (filled fold by fold)
# mu0:  predicted visit probability IF NOT emailed
# fold: the fold in which each customer was scored (used to rank within folds below)
mu1, mu0, fold = np.zeros(len(df)), np.zeros(len(df)), np.zeros(len(df), dtype=int)

# k: fold number; train: row positions of the 4 training folds combined; test: row positions of the held-out fold
for k, (train, test) in enumerate(folds.split(df, strata)):
    # treated / control: training rows split by what actually happened to the customer
    treated, control = train[is_treated[train]], train[~is_treated[train]]

    # m1: logistic regression of visit, fitted only on emailed customers      -> estimates mu_1(x)
    # m0: logistic regression of visit, fitted only on non-emailed customers  -> estimates mu_0(x)
    m1 = smf.logit(FORMULA, data=df.iloc[treated]).fit(disp=0)
    m0 = smf.logit(FORMULA, data=df.iloc[control]).fit(disp=0)

    # Both models score the held-out customers (whether they were emailed or not).
    mu1[test] = m1.predict(df.iloc[test])
    mu0[test] = m0.predict(df.iloc[test])
    fold[test] = k

df["fold"] = fold
df["uplift_score"] = mu1 - mu0                     # T-learner: predicted extra visit probability from the email
df["response_score"] = mu1                         # response model: P(visit) if emailed ("likely to visit")
df["rule_score"] = df["both"].astype(float)        # phase 3 rule: 1 for "Both" buyers, 0 otherwise

# --- Rank within folds -------------------------------------------------------
# rank(pct=True): percentile of each score within its own fold (0 = lowest, 1 = highest)
df["uplift_rank"] = df.groupby("fold")["uplift_score"].rank(pct=True)
df["response_rank"] = df.groupby("fold")["response_score"].rank(pct=True)

print(f"Predicted uplift: mean {df['uplift_score'].mean():.1%}, "
      f"from {df['uplift_score'].min():.1%} to {df['uplift_score'].max():.1%}")

Predicted uplift: mean 7.7%, from 5.7% to 14.9%


### What did the two models learn?
To read the models, we fit them once more on all customers (the cross-fitted predictions above are still used for everything else) and put their coefficients side by side.

- **emailed (m1)** and **not emailed (m0)**: the coefficients $\beta_{1,j}$ and $\beta_{0,j}$, on the log-odds scale. A positive value means a higher visit probability; $e^{\beta}$ is the factor by which the odds of visiting change for a one-unit increase in the feature.
- **difference**: $\beta_{1,j} - \beta_{0,j}$. A non-zero difference means the email changes the effect of that feature, so the uplift depends on it.
- **p (difference)**: a z-test of the difference, combining the standard errors of the two fits (they use different customers, so they are independent).

In [4]:
fit1 = smf.logit(FORMULA, data=df[is_treated]).fit(disp=0)     # emailed customers
fit0 = smf.logit(FORMULA, data=df[~is_treated]).fit(disp=0)    # non-emailed customers

coefs = pd.DataFrame({"emailed (m1)": fit1.params, "not emailed (m0)": fit0.params})
coefs["difference"] = coefs["emailed (m1)"] - coefs["not emailed (m0)"]
z = coefs["difference"] / np.sqrt(fit1.bse ** 2 + fit0.bse ** 2)    # bse: standard errors of the coefficients
coefs["p (difference)"] = 2 * stats.norm.sf(z.abs())                # two-sided p-value
coefs.round(3)

,emailed (m1),not emailed (m0),difference,p (difference)
Intercept,-1.627,-2.340,0.712,0.000
recency,-0.050,-0.067,0.018,0.042
np.log(history),0.063,0.099,-0.037,0.236
both,0.704,0.512,0.192,0.028


### Step results
- **Intercept:** the largest difference (0.71, p < 0.001). This is where most of the email effect sits: the email raises everyone's odds of visiting. (On its own, the intercept describes an impossible customer, with recency 0 and \$1 of spend, so only the difference is meaningful.)
- **both:** "Both" buyers visit more in either group, and the gap is larger with the email (0.70 vs 0.51, p = 0.03): their odds are multiplied by $e^{0.70} \approx 2.0$ when emailed, by $e^{0.51} \approx 1.7$ when not. This is the phase 3 finding again.
- **recency:** the visit probability drops with every month since the last purchase, slightly less steeply with the email (p = 0.04).
- **log(history):** bigger spenders visit a little more, with no clear difference between the groups (p = 0.24).

Apart from the intercept, the differences are small, so the predicted uplift varies little between customers: about 6 to 8 pp for most of them, and up to 15 pp for the "Both" buyers.

## Evaluation: Qini curve
**The problem.** We cannot check one customer's predicted uplift, because we never see the same customer both emailed and not emailed. We *can* check a group: within any group, the emailed and non-emailed customers are comparable (randomisation), so the difference in their visit rates is the group's real uplift.

**The idea.** A good ranking puts the customers who respond most to the email at the top. So we sort customers by their score, best first, and ask: *if we emailed only the top φ of the list, how many extra visits would the email produce?* (φ, phi, is a fraction between 0 and 100%.) The **Qini curve** plots this number for every φ.

**The formula.** Among the customers in the top φ of the list:

$$Q(\varphi) = Y_T(\varphi) - Y_C(\varphi)\,\frac{N_T(\varphi)}{N_C(\varphi)}$$

- $Y_T$, $N_T$: number of visits, and number of customers, among the **emailed** (treatment) customers in the top φ.
- $Y_C$, $N_C$: the same for the **non-emailed** (control) customers in the top φ.
- $Y_C \cdot N_T / N_C$: the visits the emailed customers would have made *without* the email (the control visit rate, scaled to the size of the emailed group). Subtracting it from $Y_T$ leaves the visits caused by the email.

We rescale $Q$ to **extra visits per 1,000 customers**, the same unit as the phase 2 headline (+77 visits per 1,000).

**How to read it.** At φ = 100% everyone is emailed, so every ranking ends at the same total (≈ +77). Random targeting gives a straight line from 0 to that total: emailing a random 30% yields 30% of the extra visits. A useful ranking bends *above* the line, because it finds the extra visits early. The **Qini coefficient** is the area between a curve and the random line, which is *the average gain over random targeting across all shares of customers emailed*, in extra visits per 1,000 customers. 0 means no better than random.

Note: the T-learner (through the `both` feature) and the buyer-type rule both use the phase 3 finding, which came from these same customers.

In [5]:
def uplift_curve(y, t, score, seed=SEED):
    """Extra outcomes per 1,000 customers when only the top fraction (by score) is emailed."""
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(score)), -np.asarray(score)))   # best first, ties in random order
    y, t = np.asarray(y, dtype=float)[order], np.asarray(t)[order]
    n_t, n_c = np.cumsum(t), np.cumsum(1 - t)                           # N_T, N_C in the top fraction
    y_t, y_c = np.cumsum(y * t), np.cumsum(y * (1 - t))                 # Y_T, Y_C in the top fraction
    q = y_t - np.divide(y_c * n_t, n_c, out=np.zeros(len(y)), where=n_c > 0)   # Qini: Y_T - Y_C * N_T / N_C
    n = np.arange(1, len(y) + 1)
    per_1000 = np.divide(q * n, n_t, out=np.zeros(len(y)), where=n_t > 0) * 1000 / len(y)
    return np.r_[0, n / len(y)], np.r_[0, per_1000]


def qini_coefficient(y, t, score):
    """Average gain over random targeting across all shares emailed, in extra outcomes per 1,000 customers."""
    frac, curve = uplift_curve(y, t, score)
    return np.trapezoid(curve - frac * curve[-1], frac)


# SCORES: the three rankings we compare (display name -> column used to sort customers)
SCORES = {"T-learner uplift": "uplift_rank", "Response model": "response_rank",
          "Buyer-type rule": "rule_score"}

### Gain over random targeting

To better visualize the difference of the Qini curves compared to the random targeting, the plots show each curve **minus** the random line instead.

- **x-axis:** the share of customers emailed, taken from the top of each ranking (%).
- **y-axis:** extra visits per 1,000 customers *beyond* what emailing the same number of random customers would give.
- **Dashed line at 0:** random targeting. Above it, the ranking beats random at that share emailed.
- **Lines:** the T-learner uplift ranking, the response model (rank by predicted visit probability if emailed) and the buyer-type rule ("Both" buyers first, then random). Hover to see the total extra visits at each share emailed.

In [6]:
COLORS = {"T-learner uplift": "#2a78d6", "Response model": "#eb6834", "Buyer-type rule": "#1baf7a"}

fig = go.Figure()
for name, col in SCORES.items():
    frac, curve = uplift_curve(df["visit"], df["t"], df[col])
    gain = curve - frac * curve[-1]                       # extra visits beyond random targeting
    idx = np.r_[0:len(frac):100, len(frac) - 1]           # every 100th point is enough to draw
    fig.add_trace(go.Scatter(
        x=frac[idx] * 100, y=gain[idx], name=name, mode="lines",
        line=dict(color=COLORS[name], width=2), customdata=curve[idx],
        hovertemplate=("Top %{x:.0f}% emailed: +%{customdata:.0f} visits per 1,000"
                       "<br>%{y:+.1f} vs random<extra>" + name + "</extra>"),
    ))
fig.add_hline(y=0, line=dict(color=GREY, width=1.5, dash="dash"),
              annotation_text="random targeting", annotation_position="bottom left",
              annotation_font_color=GREY)
fig.update_layout(
    title="Extra visits gained over random targeting (cross-fitted scores)",
    template="plotly_white", height=450,
    xaxis_title="Customers emailed, ranked by score (%)",
    yaxis_title="Extra visits per 1,000 customers, vs random",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=90, b=10),
)
fig.show()

### What we learn from the graph
- **All three rankings find the same top 10%.** At 10% every curve is about +6 extra visits per 1,000 customers above random, because they all put the "Both" buyers first.
- **Beyond 10%, the T-learner adds little.** Its gain stays roughly flat at +4 to +5 between 15% and 65%. A flat stretch means the customers added there respond like an average customer: the model cannot tell them apart. After 65% it falls to 0, as it must at 100%.
- **The buyer-type rule shows how noisy these curves are.** After the "Both" buyers its order is random by construction, so its curve should fall in a straight line from +6 to 0. The dips and bumps around that line (1-2 extra visits per 1,000) are noise.
- **The response model is highest between about 15% and 55%** (up to +7 at 30%), but the gap with the other two is about the size of that noise.
- Every curve ends at 0 at 100%: when everyone is emailed, the order no longer matters.
- **More extra visits in total does not mean more efficient.** The hover shows the total extra visits, which keep growing up to 100%. The next chart shows the efficiency: the share of emailed customers who visit because of the email.

### Efficiency: extra visits per emailed customer
The Qini chart counts the *total* extra visits per 1,000 customers. The total keeps growing as more customers are emailed, so emailing everyone gives the largest number (+77 per 1,000). That does not make it the most efficient choice. Efficiency asks a different question: *of the customers we email, what share visits because of the email?*

$$\text{efficiency}(\varphi) = \frac{Q(\varphi)}{N_T(\varphi)}$$

- $Q(\varphi)$ and $N_T(\varphi)$: the extra visits and the number of emailed customers in the top φ of the ranking, as in the Qini formula above.
- The two charts are linked: extra visits per 1,000 customers = efficiency × emails sent per 1,000 customers. For example, emailing the top 20% (200 emails per 1,000 customers) at an efficiency of 10% gives 0.10 × 200 = +20 visits per 1,000 customers.

How to read the chart:
- **x-axis:** the share of customers emailed, from the top of each ranking (%). The chart starts at 5%: below that, the top group is only a few hundred customers and the ratio is mostly noise.
- **y-axis:** the efficiency, extra visits per emailed customer (%).
- **Dashed line:** random targeting. A random group responds like the average customer whatever its size, so its efficiency is the same at every share emailed and equals that of emailing everyone (7.7%).
- **Hover:** the efficiency and the total extra visits per 1,000 customers at that share emailed (the number in the Qini chart).

In [7]:
# --- Efficiency: extra visits per emailed customer ----------------------------------------
# START: smallest share emailed that is drawn. Below ~5% the top group is a few hundred customers and the ratio is mostly noise.
START = 0.05

fig = go.Figure()
for name, col in SCORES.items():
    frac, curve = uplift_curve(df["visit"], df["t"], df[col])      # curve: extra visits per 1,000 customers
    # efficiency: extra visits per 1,000 customers / emails sent per 1,000 customers (1000 * frac)
    efficiency = np.divide(curve, 1000 * frac, out=np.zeros(len(frac)), where=frac > 0)
    idx = np.r_[np.searchsorted(frac, START):len(frac):100, len(frac) - 1]
    fig.add_trace(go.Scatter(
        x=frac[idx] * 100, y=efficiency[idx] * 100, name=name, mode="lines",
        line=dict(color=COLORS[name], width=2), customdata=curve[idx],
        hovertemplate=("Top %{x:.0f}% emailed: %{y:.1f}% of them visit because of the email"
                       "<br>= +%{customdata:.0f} visits per 1,000 customers<extra>" + name + "</extra>"),
    ))
# Random targeting has the same efficiency at every share emailed: that of emailing everyone (curve[-1] / 1000)
fig.add_hline(y=curve[-1] / 1000 * 100, line=dict(color=GREY, width=1.5, dash="dash"),
              annotation_text="random targeting (= emailing everyone)", annotation_position="bottom right",
              annotation_font_color=GREY)
fig.update_layout(
    title="Efficiency: extra visits per emailed customer, by share of customers emailed (cross-fitted scores)",
    template="plotly_white", height=450,
    xaxis=dict(title="Customers emailed, ranked by score (%)", range=[0, 100]),
    yaxis=dict(title="Extra visits per emailed customer (%)", rangemode="tozero"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=90, b=10),
)
fig.show()

### What we learn from the graph
- **Emailing everyone is the least efficient option:** +77 visits per 1,000 customers means that only 7.7% of the emailed customers visit because of the email. The total is the largest because many more emails are sent, not because each email works better.
- **Efficiency falls fastest right after the top 10%** (the "Both" buyers) and flattens after about 50%: from there on, the customers added respond close to the average.
- **The three rankings are almost indistinguishable,** as in the Qini chart; the response model is slightly ahead between about 15% and 45%.

### Qini coefficients with 95% bootstrap confidence intervals
The **bootstrap** estimates how much the Qini coefficients would change with a different sample of customers: we resample the customers with replacement 200 times, recompute every coefficient each time, and keep the middle 95% of the results as the confidence interval (CI). The scores stay fixed, so the CI covers evaluation noise, not model-fitting noise.

The "minus" rows are **paired differences**, computed on the same resample each time: if their CI excludes 0, one ranking is clearly better than the other.

In [8]:
def bootstrap_qini(df, n_boot=200, seed=SEED):
    """Qini coefficient of each ranking on resampled customers (the scores stay fixed)."""
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_boot):
        d = df.iloc[rng.integers(0, len(df), len(df))]
        rows.append({name: qini_coefficient(d["visit"], d["t"], d[col]) for name, col in SCORES.items()})
    return pd.DataFrame(rows)


qini = {name: qini_coefficient(df["visit"], df["t"], df[col]) for name, col in SCORES.items()}
boot = bootstrap_qini(df)

# PAIRS: the paired differences we test (first minus second)
PAIRS = [("T-learner uplift", "Response model"), ("T-learner uplift", "Buyer-type rule"),
         ("Response model", "Buyer-type rule")]
for a, b in PAIRS:
    label = f"{a} minus {b.lower()}"
    qini[label] = qini[a] - qini[b]
    boot[label] = boot[a] - boot[b]

qini_table = pd.DataFrame({"Qini coefficient": pd.Series(qini),
                           "95% CI low": boot.quantile(0.025),
                           "95% CI high": boot.quantile(0.975)})
qini_table.round(2)

,Qini coefficient,95% CI low,95% CI high
T-learner uplift,3.27,1.46,5.34
Response model,3.92,1.97,5.99
Buyer-type rule,3.14,0.98,5.03
T-learner uplift minus response model,-0.65,-1.72,0.26
T-learner uplift minus buyer-type rule,0.13,-1.81,2.13
Response model minus buyer-type rule,0.77,-1.31,3.10


### What we learn from the table
All values are in extra visits per 1,000 customers, averaged over every share emailed.

- **All three rankings beat random targeting:** every CI is above 0. The T-learner scores 3.27 (CI 1.46 to 5.34)
- **The T-learner ties the buyer-type rule:** the difference is +0.13 (CI −1.81 to 2.13).
- **It scores slightly below the response model:** −0.65 (CI −1.72 to 0.26), not statistically clear.
- **The response model scores highest (3.92),** but its edge over the buyer-type rule (+0.77, CI −1.31 to 3.10) is not statistically clear either.

So none of the rankings is clearly better than the others: the T-learner adds nothing over the two simpler ones.

### Uplift by decile
This chart checks the T-learner against what actually happened. Customers are split into 10 equal groups (**deciles**) by their T-learner score: decile 1 holds the 10% with the highest predicted uplift, decile 10 the lowest. In each decile we measure the real lift with the phase 2 `proportion_test`.

- **Bars:** the observed lift in the decile, in percentage points (pp): visit rate of the emailed customers minus visit rate of the non-emailed customers.
- **Error bars:** the 95% confidence interval of that observed lift.
- **Black ticks:** the average uplift the model *predicted* for the decile.

If the model ranks well, the bars fall from left to right. If its predictions are accurate (*calibrated*), each tick sits inside its error bar.

In [9]:
# decile: 1 = top 10% of the T-learner ranking, 10 = bottom 10%
df["decile"] = pd.qcut(df["uplift_rank"].rank(method="first", ascending=False), 10, labels=range(1, 11))

rows = []
for decile, sub in df.groupby("decile", observed=True):
    r = proportion_test(values(sub, "visit", "mens_email"), values(sub, "visit", "no_email"))
    rows.append({"decile": int(decile), "predicted_uplift": sub["uplift_score"].mean(), **r})
deciles = pd.DataFrame(rows)

lift, low, high = (deciles[c].to_numpy() * 100 for c in ("abs_lift", "abs_ci_low", "abs_ci_high"))
fig = go.Figure()
fig.add_trace(go.Bar(
    x=deciles["decile"], y=lift, name="Observed lift (95% CI)",
    marker=dict(color="#2a78d6", cornerradius=4),
    error_y=dict(type="data", symmetric=False, array=high - lift, arrayminus=lift - low,
                 color="#52514e", thickness=1.5, width=3),
    hovertemplate="Decile %{x}: %{y:+.1f} pp observed<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=deciles["decile"], y=deciles["predicted_uplift"] * 100, name="Predicted uplift", mode="markers",
    marker=dict(color="#0b0b0b", size=14, symbol="line-ew-open", line=dict(width=2)),
    hovertemplate="Decile %{x}: %{y:+.1f} pp predicted<extra></extra>",
))
fig.update_layout(
    title="Observed visit lift by decile of predicted uplift (1 = highest)",
    template="plotly_white", height=420, bargap=0.3,
    xaxis=dict(title="Decile of predicted uplift", tickmode="linear", dtick=1),
    yaxis=dict(title="Visit lift, men's email vs no email (pp)", zeroline=True, zerolinecolor=GREY),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=90, b=10),
)
fig.show()

### What we learn from the graph
- **Decile 1 works, and it is exactly the "Both" buyers** (see the table below). The model predicted +13.7 pp and the observed lift is +13.8 pp.
- **Deciles 2-10 are flat.** The observed lift moves between 6 and 8 pp with no trend from left to right, so the ranking inside the remaining 90% of customers does not find stronger or weaker responders.
- **The predictions are calibrated:** every black tick sits inside its error bar. The model is not wrong; it simply predicts almost the same uplift (6.3 to 7.7 pp) for everyone outside decile 1, and the data agree.

What we gain from this graph: all the useful information in the T-learner is a single split, "Both" buyers vs everyone else, which is the phase 3 rule. Recency and spend shift the predictions a little, but the observed lift does not follow them.

### Who gets targeted?
The table compares the top 10%, the middle 80% and the bottom 10% of the T-learner ranking. Each value is an average over the customers in the group: `recency` in months, `history` in \$, and the other rows are shares (0.62 = 62% of the group).

In [10]:
group = np.select([df["decile"] == 1, df["decile"] == 10], ["Top 10%", "Bottom 10%"], "Middle 80%")
profile = (pd.get_dummies(df[["recency", "history", "newbie", "buyer_type", "channel"]], dtype=float)
           .groupby(group).mean()
           .reindex(["Top 10%", "Middle 80%", "Bottom 10%"]).T)
profile.round(2)

,Top 10%,Middle 80%,Bottom 10%
recency,4.66,5.25,10.94
history,532.50,219.10,133.27
newbie,0.61,0.49,0.49
buyer_type_Both,1.00,0.00,0.00
buyer_type_Men's only,0.00,0.50,0.49
buyer_type_Women's only,0.00,0.50,0.51
channel_Multichannel,0.31,0.11,0.05
channel_Phone,0.34,0.44,0.48
channel_Web,0.35,0.45,0.47


### What we learn from the table
- **The top 10% are all "Both" buyers** (1.00, against 0.00 elsewhere): the T-learner's best decile is exactly the phase 3 segment. They also spend more (\$533 vs \$219) and buy more often through several channels (31% vs 11% multichannel), which comes with buying across both categories.
- **The bottom 10% are customers whose last purchase was long ago** (10.9 months, against about 5) and who spent little (\$133). The model gives them the lowest uplift because they rarely visit at all. Yet their observed lift (+7.5 pp in decile 10) is no lower than in the middle deciles: in percentage points, the email works about as well on them.

## Targeting economics
For a ranking, emailing the top X% of customers gives, per 1,000 customers:

$$\text{Profit}(X) = m \cdot R(X) - c \cdot 1000 \cdot X$$

- $R(X)$: the extra revenue caused by the email when the top X% is emailed: the Qini curve computed on `spend` instead of `visit` (noisier, because few customers buy).
- $m$: the margin, the share of revenue that is profit.
- $c$: the cost of sending one email; $1000 \cdot X$ is the number of emails sent per 1,000 customers.

Assumptions: margin 50%, and email costs of \$0.15 / \$0.30 / \$0.50.

In [11]:
MARGIN = 0.50                       # assumed share of revenue that is profit
COSTS = [0.15, 0.30, 0.50]          # assumed cost per email ($): low / mid / high

### Profit curves
Profit per 1,000 customers when the top X% of the T-learner ranking is emailed, $\text{Profit}(X) = m \cdot R(X) - c \cdot 1000 \cdot X$, for four email costs $c$: free (\$0) and the three costs above.

- **x-axis:** share of customers emailed, from the top of the ranking (%).
- **y-axis:** profit in \$ per 1,000 customers.
- **Lines:** one per email cost, from black (free) through dark blue (cheap) to light blue (expensive). Free emails cost nothing, so that line is just $m \cdot R(X)$. Every other line sits below it by the cost of the emails sent, $c \cdot 1000 \cdot X$, so the gap grows with the share emailed.

**How to read a line**
- **What it adds up:** $R(X)$ is the extra revenue caused by the email when the top X% is emailed: what the emailed customers in that group spend, minus what they would have spent without the email (estimated from the not-emailed customers in the same group). The margin $m$ = 50% keeps the part of it that is profit, and then the email costs are subtracted.
- **Running total:** the value at X% is the profit from everyone in the top X%, not from the customers at X%. With free emails, for example, emailing the top 25% brings in \$172 per 1,000 customers, and emailing everyone \$385 (half of \$770 of extra sales).
- **Slope: the profit of the next emails, per email.** It is what the next emails earn before their cost, minus $c$. Before costs (the slope of the free line), the next emails earn about \$0.92 each in the top 10%, about \$0.15 between 25% and 50%, and about \$0.35 after 50%. A line rises where the next emails earn more than they cost and falls where they earn less. For example, at \$0.30 per email between 25% and 50%: \$0.15 − \$0.30 = −\$0.15 per email, so profit falls.
- **Highest point:** the most profitable share to email at that cost; the printed lines above the chart give it.
- **Break-even** (profit = 0) is where a line crosses \$0. Dividing the free line by the number of emails sent gives what an email earns on average before its cost. It is lowest when everyone is emailed (\$0.38), so a cost below \$0.38 makes a profit whatever share is emailed, while a higher cost turns into a loss once too many customers are emailed.

**Smoothing.** $R(X)$ is noisy: only about 1 customer in 100 buys, so a few large orders create peaks that are not real. The profit curves are therefore smoothed with a Gaussian kernel. Each point becomes a weighted average of its neighbours, with weights following a normal curve whose standard deviation is $\sigma$ = 5% of customers (`SIGMA`). Bumps narrower than about $\sigma$ are averaged out. Both ends stay fixed, so emailing nobody (\$0) and emailing everyone keep their exact values.

`RANKING` sets which ranking is drawn (`"response_rank"` or `"rule_score"` work too). For `"rule_score"` the calculated line of the next subsection is drawn instead of a smoothed one.

In [12]:
RANKING = "uplift_rank"                                       # which ranking to draw
EMAIL_COSTS = [0.0] + COSTS                                   # $0 = free emails, then the three costs above
RAMP = ["#0b0b0b", "#104281", "#2a78d6", "#86b6ef"]           # darkest to lightest = free to expensive
SIGMA = 0.05                                                  # smoothing width: sd of the Gaussian kernel, as a share of customers


def smooth(curve, sigma):
    """Gaussian smoothing; odd reflection at the edges keeps both end points unchanged."""
    pad = 4 * sigma                                           # the kernel reaches 4 sd on each side
    padded = np.pad(curve, pad, mode="reflect", reflect_type="odd")
    return gaussian_filter1d(padded, sigma)[pad:-pad]


def block_line(frac, curve, score):
    """Expected curve of a ranking with tied scores: a straight line within each block of equal scores.

    Customers with the same score are emailed in random order, so on average each extra email in a block adds
    that block's average: only the block ends carry information.
    """
    shares = (score.value_counts().sort_index(ascending=False).cumsum() / len(score)).to_numpy()
    knots = np.r_[0.0, shares]                                # 0, end of block 1, ..., 1
    return np.interp(frac, knots, np.interp(knots, frac, curve))


frac, revenue = uplift_curve(df["spend"], df["t"], df[RANKING])
idx = np.r_[0:len(frac):100, len(frac) - 1]
x = frac[idx] * 100

fig = go.Figure()
for cost, color in zip(EMAIL_COSTS, RAMP):
    label = "free emails" if cost == 0 else f"${cost:.2f} per email"
    if RANKING == "rule_score":     # random order after the "Both" buyers: draw the calculated line (see below)
        profit = MARGIN * block_line(frac, revenue, df[RANKING]) - cost * 1000 * frac
    else:
        profit = smooth(MARGIN * revenue - cost * 1000 * frac, int(SIGMA * len(frac)))
    best = np.argmax(profit)
    print(f"{label}: best to email the top {frac[best]:.0%} "
          f"(${profit[best]:.0f} vs ${profit[-1]:.0f} emailing everyone, per 1,000 customers)")
    fig.add_trace(go.Scatter(
        x=x, y=profit[idx], name=f"Profit, {label}", mode="lines",
        line=dict(color=color, width=2),
        hovertemplate="Top %{x:.0f}%: $%{y:.0f} profit per 1,000 customers<extra>" + label + "</extra>",
    ))
fig.update_layout(
    title=f"Profit by share of customers emailed (margin {MARGIN:.0%}, "
          + ("calculated)" if RANKING == "rule_score" else "smoothed)"),
    template="plotly_white", height=480,
    xaxis=dict(title="Customers emailed, ranked by score (%)", range=[0, 100]),
    yaxis=dict(title="Profit per 1,000 customers ($)", zeroline=True, zerolinecolor=GREY),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=100, b=10),
)
fig.show()

free emails: best to email the top 100% ($385 vs $385 emailing everyone, per 1,000 customers)


$0.15 per email: best to email the top 100% ($235 vs $235 emailing everyone, per 1,000 customers)


$0.30 per email: best to email the top 25% ($97 vs $85 emailing everyone, per 1,000 customers)


$0.50 per email: best to email the top 19% ($54 vs $-115 emailing everyone, per 1,000 customers)


### What we learn from the graph
- **At \$0.50 per email there is a break-even point:** the line crosses \$0 at the top 39%. Emailing more customers than that loses money, and emailing everyone loses \$115 per 1,000 customers. The reason is that \$0.50 is more than an average email earns before its cost (\$0.38).
- **At \$0.15 and \$0.30 per email there is none:** both costs are below \$0.38, so every share emailed makes a profit.
- **Most profitable share to email,** per 1,000 customers:
  - free and \$0.15: everyone (\$385 and \$235), since profit keeps rising up to 100%;
  - \$0.30: the top 25% (\$97), only \$12 more than emailing everyone (\$85);
  - \$0.50: the top 19% (\$54), against a loss of \$115 for emailing everyone.

### The simple rule: a calculated line
The simple rule emails the "Both" buyers first and then everyone else in random order. Measured like the curves above, its profit curve follows one particular random order and picks up its noise, so its peak can land almost anywhere.

Its expected curve, however, can be calculated. Inside each block (the "Both" buyers, then everyone else) the order is random, so on average each extra email brings in that block's average. The curve is therefore two straight lines, fixed by what an email earns before its cost in each block:

$$e_{\text{Both}} = \frac{m \cdot R(s)}{1000 \cdot s}, \qquad e_{\text{rest}} = \frac{m \cdot \left(R(1) - R(s)\right)}{1000 \cdot (1 - s)}$$

- $s$: the share of customers who are "Both" buyers; the two lines meet at $X = s$.
- $R(s)$ and $R(1)$: the extra revenue per 1,000 customers when only the "Both" buyers, or everyone, are emailed.
- $e_{\text{Both}}$, $e_{\text{rest}}$: what one email earns before its cost, in each block.

The chart compares the measured (smoothed) and the calculated profit line at \$0.30 per email.

In [13]:
COST = 0.30                                                   # email cost for the comparison chart

x_rule, rev_rule = uplift_curve(df["spend"], df["t"], df["rule_score"])
s = df["both"].mean()                                         # share of "Both" buyers
r_s = np.interp(s, x_rule, rev_rule)                          # R(s): extra revenue when only they are emailed
e_both = MARGIN * r_s / (1000 * s)
e_rest = MARGIN * (rev_rule[-1] - r_s) / (1000 * (1 - s))
print(f'"Both" buyers: {s:.1%} of customers. Before its cost, an email earns ${e_both:.2f} when sent to a '
      f'"Both" buyer and ${e_rest:.2f} when sent to anyone else.')

measured = smooth(MARGIN * rev_rule - COST * 1000 * x_rule, int(SIGMA * len(x_rule)))
calculated = MARGIN * block_line(x_rule, rev_rule, df["rule_score"]) - COST * 1000 * x_rule
for name, profit in [("Measured", measured), ("Calculated", calculated)]:
    best = np.argmax(profit)
    print(f"{name}: best to email the top {x_rule[best]:.0%} (${profit[best]:.0f} per 1,000 customers)")

idx = np.r_[0:len(x_rule):100, len(x_rule) - 1]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_rule[idx] * 100, y=measured[idx], name="Measured (smoothed)", mode="lines",
    line=dict(color=GREY, width=2, dash="dot"),
    hovertemplate="Top %{x:.0f}%: $%{y:.0f} measured<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=x_rule[idx] * 100, y=calculated[idx], name="Calculated", mode="lines",
    line=dict(color=COLORS["Buyer-type rule"], width=2.5),
    hovertemplate="Top %{x:.0f}%: $%{y:.0f} calculated<extra></extra>",
))
fig.update_layout(
    title=f"Simple rule: measured vs calculated profit (${COST:.2f} per email, margin {MARGIN:.0%})",
    template="plotly_white", height=420,
    xaxis=dict(title="Customers emailed, ranked by score (%)", range=[0, 100]),
    yaxis=dict(title="Profit per 1,000 customers ($)", zeroline=True, zerolinecolor=GREY),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=10, r=10, t=100, b=10),
)
fig.show()

"Both" buyers: 10.2% of customers. Before its cost, an email earns $0.91 when sent to a "Both" buyer and $0.33 when sent to anyone else.


Measured: best to email the top 47% ($104 per 1,000 customers)
Calculated: best to email the top 100% ($85 per 1,000 customers)


### What we learn from the graph
- **Two numbers define the rule:** before its cost, an email earns \$0.91 when sent to a "Both" buyer (10.2% of customers) and \$0.33 when sent to anyone else.
- **The measured curve wanders around the calculated line.** At \$0.30 per email it peaks at the top 47% (\$104), but that peak comes from one particular random order: after the "Both" buyers the rule picks customers at random, and no stretch of a random order is better than another. The calculated line rises slowly after the corner, because \$0.33 is just above the \$0.30 cost, so its best is emailing everyone (\$85).
- **The best choice for the rule is always one of three:**
  - email everyone if an email costs less than \$0.33;
  - only the "Both" buyers if it costs between \$0.33 and \$0.91;
  - nobody if it costs more than \$0.91.

  Between \$0.33 and \$0.38, emailing everyone still makes a profit (an average email earns \$0.38), but emailing only the "Both" buyers makes more.

## Conclusion
**Question:** can a model that ranks individual customers by the email's effect target better than the phase 3 rule ("Both" buyers first)?

**Answer: no. In this case the T-learner does not improve the visit uplift.**
- **Ranking quality:** all three rankings beat random targeting, with Qini coefficients of 3.3 (T-learner), 3.9 (response model) and 3.1 (buyer-type rule) extra visits per 1,000 customers. None of them is statistically better than the others.
- **Why:** the email's effect barely varies between customers. It raises everyone's odds of visiting (the intercept in the coefficient table), and only the "Both" buyers respond clearly more: +13.8 pp in decile 1, against 6-8 pp in every other decile, whatever their recency or spend. The T-learner finds this split and nothing more, so it reduces to the phase 3 rule.
- **Efficiency:** targeting the top of the ranking makes each email work harder. In the top 10%, about 14% of the emailed customers visit because of the email, against 7.7% when everyone is emailed. The total number of extra visits, however, is largest when everyone is emailed.
- **Economics:** on average an email earns about \$0.38 before its cost (half of the \$0.77 of extra sales per emailed customer). Below that cost, emailing makes a profit at every share emailed: at \$0.15 per email, emailing everyone gives the most profit, and at \$0.30, emailing the top 25% beats emailing everyone by only \$12 per 1,000 customers, well within the noise of spend. Above it, emailing too many customers loses money: at \$0.50 per email the break-even point is at the top 39%, emailing everyone loses \$115 per 1,000 customers, and emailing the top 19% earns \$54. For the simple rule the choice can be calculated exactly: an email earns \$0.91 when sent to a "Both" buyer and \$0.33 when sent to anyone else, so at \$0.50 it pays to email only the "Both" buyers (\$42). The uplift model's small extra comes from customers it ranks no better than at random (deciles 2-10 are flat), so it is most likely noise.

**Recommendation**
- **If an email costs less than about \$0.33, email everyone.** That is what an email earns outside the "Both" buyers, so every email still pays for itself.
- **If it costs between \$0.33 and \$0.91, email only the "Both" buyers** (10% of customers). The phase 3 rule finds them without a model to build and maintain: the uplift model's top 10% are the same customers.
- **Above \$0.91 per email, don't email.**